In [4]:
from typing import TypedDict, Optional
from langgraph.graph import StateGraph, END

In [2]:
class SimpleState(TypedDict):
    count: int

def add_one(state: SimpleState) -> SimpleState:
    print("Running add_one, current count:", state["count"])
    return {"count": state["count"] + 1}

graph = StateGraph(SimpleState)
graph.add_node("add_one", add_one)
graph.set_entry_point("add_one")
graph.add_edge("add_one", END)

app = graph.compile()

result = app.invoke({"count": 0})
print("Final result:", result)

Running add_one, current count: 0
Final result: {'count': 1}


In [5]:
class SimpleState(TypedDict):
    count: int
    flagged: Optional[bool]

def check_value(state: SimpleState) -> SimpleState:
    print("Checking value:", state["count"])
    if state["count"] > 5:
        return {"flagged": True}
    else:
        return {"flagged": False}

def handle_flag(state: SimpleState) -> SimpleState:
    print("Handling flagged case for count:", state["count"])
    return {"count": state["count"] * 10}

def route(state: SimpleState) -> str:
    if state["flagged"]:
        return "handle_flag"
    else:
        return END

graph = StateGraph(SimpleState)
graph.add_node("check_value", check_value)
graph.add_node("handle_flag", handle_flag)

graph.set_entry_point("check_value")
graph.add_conditional_edges("check_value", route)
graph.add_edge("handle_flag", END)

app = graph.compile()

print("--- Test 1: count = 3 ---")
result1 = app.invoke({"count": 3, "flagged": None})
print("Result:", result1)

print("--- Test 2: count = 10 ---")
result2 = app.invoke({"count": 10, "flagged": None})
print("Result:", result2)

--- Test 1: count = 3 ---
Checking value: 3
Result: {'count': 3, 'flagged': False}
--- Test 2: count = 10 ---
Checking value: 10
Handling flagged case for count: 10
Result: {'count': 100, 'flagged': True}
